In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import statsmodels.api as sm
import statsmodels.formula.api as smf
import statsmodels.stats.api as sms
from pathlib import Path
# Display with better formatting - all columns visible
pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', None)
pd.set_option('display.width', None)
pd.set_option('display.max_colwidth', None)

In [2]:

print(Path.cwd())
# Gets the parent directory of the current working directory
project_root = Path.cwd().parent.parent

print(project_root)

/Users/gkumargaur/workspace/datascience/boston_university/datascience-practice/sas_hackthon/stats_ml
/Users/gkumargaur/workspace/datascience/boston_university/datascience-practice


In [3]:
fullpath = "{}/data/VST152/pva_donors.csv".format(project_root)
donor_data = pd.read_csv(fullpath)


In [4]:
# Method 3: Grouped by data type
print("\n\nMethod 3: Grouped by Data Type")
print("Numeric Columns:")
numeric_cols = donor_data.select_dtypes(include=['number']).columns.tolist()
for i, col in enumerate(numeric_cols, 1):
    print(f"  {i:2d}. {col}")





Method 3: Grouped by Data Type
Numeric Columns:
   1. ID
   2. GiftCnt36
   3. GiftCntAll
   4. GiftCntCard36
   5. GiftCntCardAll
   6. GiftAvgLast
   7. GiftAvg36
   8. GiftAvgAll
   9. GiftAvgCard36
  10. GiftTimeLast
  11. GiftTimeFirst
  12. PromCnt12
  13. PromCnt36
  14. PromCntAll
  15. PromCntCard12
  16. PromCntCard36
  17. PromCntCardAll
  18. StatusCatStarAll
  19. DemCluster
  20. DemAge
  21. DemMedHomeValue
  22. DemPctVeterans
  23. DemMedIncome
  24. Donation_Amt
  25. Rep_DemMedIncome
  26. IM_Rep_DemMedIncome
  27. log_GiftCnt36


In [5]:
print("\nObject/String Columns:")
object_cols = donor_data.select_dtypes(include=['object']).columns.tolist()
for i, col in enumerate(object_cols, 1):
    print(f"  {i:2d}. {col}")




Object/String Columns:
   1. StatusCat96NK
   2. DemGender
   3. DemHomeOwner
   4. Response


In [6]:
# Disable Jupyter output truncation completely
from IPython.display import display
import sys

# Set very high limits
pd.set_option('display.max_rows', 1000)
pd.set_option('display.max_columns', 1000)
pd.set_option('display.width', 2000)
pd.set_option('display.max_colwidth', 1000)

# Create column info
col_info = pd.DataFrame({
    'Column #': range(1, len(donor_data.columns) + 1),
    'Column Name': donor_data.columns,
    'Data Type': donor_data.dtypes.values,
    'Non-Null Count': donor_data.count().values,
    'Missing': donor_data.isnull().sum().values
})

# Display without truncation
display(col_info)

,Column #,Column Name,Data Type,Non-Null Count,Missing
0,1,ID,int64,9686,0
1,2,GiftCnt36,float64,9686,0
2,3,GiftCntAll,float64,9686,0
3,4,GiftCntCard36,float64,9686,0
4,5,GiftCntCardAll,float64,9686,0
5,6,GiftAvgLast,float64,9686,0
6,7,GiftAvg36,float64,9686,0
7,8,GiftAvgAll,float64,9686,0
8,9,GiftAvgCard36,float64,7906,1780
9,10,GiftTimeLast,float64,9686,0


In [7]:
# Method 5: Compact grid format
print("\n\nMethod 5: Compact Grid")
cols_per_row = 4
for i in range(0, len(donor_data.columns), cols_per_row):
    row = donor_data.columns[i:i+cols_per_row].tolist()
    print("  |  ".join(f"{col:20s}" for col in row))



Method 5: Compact Grid
ID                    |  GiftCnt36             |  GiftCntAll            |  GiftCntCard36       
GiftCntCardAll        |  GiftAvgLast           |  GiftAvg36             |  GiftAvgAll          
GiftAvgCard36         |  GiftTimeLast          |  GiftTimeFirst         |  PromCnt12           
PromCnt36             |  PromCntAll            |  PromCntCard12         |  PromCntCard36       
PromCntCardAll        |  StatusCat96NK         |  StatusCatStarAll      |  DemCluster          
DemAge                |  DemGender             |  DemHomeOwner          |  DemMedHomeValue     
DemPctVeterans        |  DemMedIncome          |  Response              |  Donation_Amt        
Rep_DemMedIncome      |  IM_Rep_DemMedIncome   |  log_GiftCnt36       


In [8]:
donor_data.head()

,ID,GiftCnt36,GiftCntAll,GiftCntCard36,GiftCntCardAll,GiftAvgLast,GiftAvg36,GiftAvgAll,GiftAvgCard36,GiftTimeLast,GiftTimeFirst,PromCnt12,PromCnt36,PromCntAll,PromCntCard12,PromCntCard36,PromCntCardAll,StatusCat96NK,StatusCatStarAll,DemCluster,DemAge,DemGender,DemHomeOwner,DemMedHomeValue,DemPctVeterans,DemMedIncome,Response,Donation_Amt,Rep_DemMedIncome,IM_Rep_DemMedIncome,log_GiftCnt36
0,14974,2.0,4.0,1.0,3.0,17.0,13.50,9.25,17.00,21.0,66.0,8.0,17.0,26.0,3.0,8.0,13.0,A,0.0,0,NaN,F,U,0.0,0.0,0.0,No,NaN,NaN,53513.457361,0.693147
1,6294,1.0,8.0,0.0,3.0,20.0,20.00,15.88,NaN,26.0,92.0,14.0,35.0,79.0,5.0,5.0,24.0,A,0.0,23,67.0,F,U,186800.0,85.0,0.0,No,NaN,NaN,53513.457361,0.000000
2,46110,6.0,41.0,3.0,20.0,6.0,5.17,3.73,5.00,18.0,111.0,12.0,23.0,51.0,5.0,11.0,22.0,S,1.0,0,NaN,M,U,87600.0,36.0,38750.0,Yes,55.451774,38750.0,38750.000000,1.791759
3,185937,3.0,12.0,3.0,8.0,10.0,8.67,8.50,8.67,9.0,93.0,14.0,22.0,44.0,2.0,6.0,16.0,E,1.0,0,NaN,M,U,139200.0,27.0,38942.0,Yes,92.103404,38942.0,38942.000000,1.098612
4,29637,1.0,1.0,1.0,1.0,20.0,20.00,20.00,20.00,21.0,21.0,10.0,15.0,13.0,4.0,7.0,6.0,F,0.0,35,53.0,M,U,168100.0,37.0,71509.0,No,NaN,71509.0,71509.000000,0.000000


## Summary Statistics

Create comprehensive summary statistics for the data, **excluding the target variable**.

### Metrics to Calculate

| Category | Metrics |
|----------|---------|
| **Central Tendency** | Mean, Mode, Median |
| **Dispersion** | Standard Deviation, Min, Max |
| **Quartiles** | Q1 (25th), Median (50th), Q3 (75th) |
| **Shape** | Skewness, Kurtosis |
| **Data Quality** | Number of Observations, Missing Values |

---

## Why is Kurtosis Useful?

Knowing the average or variance isn't enough in the real world. **Kurtosis warns you about "Black Swan" events** — extreme, unexpected occurrences.

### Real-World Applications

| Domain | Why Kurtosis Matters |
|--------|---------------------|
| **Finance & Risk Management** | A stock with *high kurtosis* has a higher chance of sudden, massive crashes or jumps. A stock with *low kurtosis* is more predictable and safer. |
| **Engineering & Manufacturing** | When testing material strength (e.g., bridges), you want *low kurtosis* (thin tails). High kurtosis means some pieces might be dangerously weak, risking sudden failure. |
| **Data Science & AI** | Calculating kurtosis helps quickly identify extreme outliers that could skew model predictions during preprocessing. |

In [9]:
# Explore potential target variables
print("=" * 60)
print("EXPLORING TARGET VARIABLES")
print("=" * 60)

print("\n1. StatusCat96NK - Unique Values:")
print(donor_data['StatusCat96NK'].value_counts())
print(f"   Data Type: {donor_data['StatusCat96NK'].dtype}")

print("\n2. StatusCatStarAll - Unique Values:")
print(donor_data['StatusCatStarAll'].value_counts())
print(f"   Data Type: {donor_data['StatusCatStarAll'].dtype}")

print("\n" + "=" * 60)
print("RECOMMENDATION:")
print("=" * 60)
print("StatusCat96NK appears to be the PRIMARY TARGET")
print("(Categories: A, E, S likely represent donor response status)")
print("StatusCatStarAll appears to be SECONDARY (binary: 0.0 or 1.0)")
print("=" * 60)

EXPLORING TARGET VARIABLES

1. StatusCat96NK - Unique Values:
StatusCat96NK
A    5826
S    2365
F     660
N     574
E     227
L      34
Name: count, dtype: int64
   Data Type: object

2. StatusCatStarAll - Unique Values:
StatusCatStarAll
1.0    5236
0.0    4450
Name: count, dtype: int64
   Data Type: float64

RECOMMENDATION:
StatusCat96NK appears to be the PRIMARY TARGET
(Categories: A, E, S likely represent donor response status)
StatusCatStarAll appears to be SECONDARY (binary: 0.0 or 1.0)


In [10]:
# Create comprehensive summary statistics (excluding target and ID)
from scipy import stats

# Exclude ID, StatusCat96NK (target), and StatusCatStarAll
exclude_cols = ['ID', 'StatusCat96NK', 'StatusCatStarAll', 'Response', 'Donation_Amt', 'DemGender', 'DemHomeOwner', 'Rep_DemMedIncome', 'IM_Rep_DemMedIncome', 'log_GiftCnt36']
numeric_cols = donor_data.select_dtypes(include=[np.number]).columns
feature_cols = [col for col in numeric_cols if col not in exclude_cols]

# Create summary statistics dataframe
summary_stats = pd.DataFrame()

for col in feature_cols:
    summary_stats[col] = {
        'Count': donor_data[col].count(),
        'Missing': donor_data[col].isna().sum(),
        'Mean': donor_data[col].mean(),
        'Std Dev': donor_data[col].std(),
        'Min': donor_data[col].min(),
        'Q1 (25%)': donor_data[col].quantile(0.25),
        'Median (50%)': donor_data[col].quantile(0.50),
        'Q3 (75%)': donor_data[col].quantile(0.75),
        'Max': donor_data[col].max(),
        'Mode': donor_data[col].mode()[0] if len(donor_data[col].mode()) > 0 else np.nan,
        'Skewness': stats.skew(donor_data[col].dropna()),
        'Kurtosis': stats.kurtosis(donor_data[col].dropna())
    }

# Transpose for better readability
summary_stats = summary_stats.T
#print(summary_stats.round(4))

In [11]:
# Display with better formatting - all columns visible
pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', None)
pd.set_option('display.width', None)
pd.set_option('display.max_colwidth', None)

# Display the summary statistics
print("\n" + "=" * 200)
print("COMPREHENSIVE SUMMARY STATISTICS (Excluding Target Variables & Engineered Features)")
print("=" * 200 + "\n")
print(summary_stats.round(4).to_string())
print("\n" + "=" * 200)



COMPREHENSIVE SUMMARY STATISTICS (Excluding Target Variables & Engineered Features)

                  Count  Missing         Mean     Std Dev    Min  Q1 (25%)  Median (50%)  Q3 (75%)       Max  Mode  Skewness  Kurtosis
GiftCnt36        9686.0      0.0       3.2055      2.1334   0.00      2.00          3.00       4.0      16.0   2.0    1.2882    2.0457
GiftCntAll       9686.0      0.0      10.5076      8.9934   1.00      4.00          8.00      15.0      91.0   1.0    1.8628    6.0440
GiftCntCard36    9686.0      0.0       1.8566      1.5954   0.00      1.00          1.00       3.0       9.0   1.0    1.1723    1.4935
GiftCntCardAll   9686.0      0.0       5.5825      4.7369   0.00      2.00          4.00       8.0      41.0   1.0    1.3311    2.0232
GiftAvgLast      9686.0      0.0      16.0177     12.0418   0.00     10.00         15.00      20.0     450.0  15.0    9.9174  245.9228
GiftAvg36        9686.0      0.0      14.8762     10.0570   0.00      9.60         13.50      18.5     2

## Key Insights: Understanding Extreme Kurtosis & Skewness

### 1. The GiftAvgLast Anomaly: Extreme Kurtosis (52,894.91) and Skewness (207.18)

Let's examine the **GiftAvgLast** variable (average value of the last gift):

| Metric | Value |
|--------|-------|
| **Min** | $0.00 |
| **Median (50%)** | $15.00 |
| **Q3 (75%)** | $20.00 |
| **Max** | $10,000.00 |
| **Skewness** | 207.18 |
| **Kurtosis** | 52,894.91 |

#### What's Happening?

**🔴 Massive Positive Skewness (207.18)**
- Normally, skewness between -0.5 and +0.5 indicates symmetry
- A skewness of **207 is astronomically high**
- Almost all donors cluster at the bottom ($10–$20), but the distribution stretches incredibly far right due to a few massive donations

**🔴 Astronomical Excess Kurtosis (52,894.91)**
- Standard normal distribution has kurtosis = 0
- A kurtosis of **52,894 is extreme "Leptokurtic"** (very peaked with heavy tails)
- Your distribution looks like a **miles-tall needle** at the center ($15 is mode and median) with massive, extreme tails
- The $10,000 maximum gift acts as a heavy anchor, pulling the kurtosis sky-high

---

### 2. The Donor Story: Three Distinct Segments

#### 👥 The "Bread and Butter" Donors (The Peak)
- **75% of donors** (Q3) give **$20 or less**
- **50% of donors** (Median) give **$15 or less**
- These are your standard, everyday supporters

#### 💎 The "Major Philanthropists" (The Tail)
- Maximum gift: **$10,000**
- Represent **< 1% of your data**
- Wildly distort mathematical averages
- Mean ($17.47) is higher than what 50% of people give—dragged up by the $10k outlier

#### ⏰ Campaign Fatigue: Timing Patterns

| Variable | Median | Kurtosis | Interpretation |
|----------|--------|----------|-----------------|
| **GiftTimeLast** | 18 months ago | 2.3 (low) | Tightly grouped—recent donations are concentrated |
| **GiftTimeFirst** | 64 months ago | -0.85 (negative) | Flat, broad curve—first donations spread widely over years |

**Insight:** Donors made their first gift over a wide timespan, but recent gifts cluster tightly around 18 months ago.
Let's zoom in on the row for GiftAvgLast (the average value of the last gift a donor gave):

Min: $0.00

Median (50%): $15.00

Q3 (75%): $20.00

Max: $10,000.00

Skewness: 207.18

Kurtosis: 52,894.91

What is happening here?
Massive Positive Skewness (207.18):
Normally, a skewness between $-0.5$ and $+0.5$ is symmetrical. A skewness of 207 is astronomically high. It tells you that almost all of your donors are clustered at the bottom (giving $10 to $20), but the distribution is stretched incredibly far to the right by a few massive, high-value donations.

Astronomical Excess Kurtosis (52,894.91):
Remember, a standard normal distribution has a kurtosis of 0. A kurtosis of 52,894 is an extreme "Leptokurtic" shape.
It indicates that your distribution looks like a miles-tall needle in the middle ($15 is the mode and median) with massive, heavy, extreme "tails."
The tail is held down by that singular $10,000 max gift (and likely a few other giant gifts), which acts as a heavy anchor pulling the kurtosis calculation sky-high.

💡 2. The Story This Tells About Your Donors
Looking across the variables, you can group your donors into a clear narrative:

The "Bread and Butter" Donors (The Peak):
Look at GiftAvgLast. 75% of your donors (Q3) give $20 or less, and half of them (Median) give $15 or less. This represents your standard, everyday supporters.

The "Major Philanthropists" (The Tail):
The maximum gift is $10,000. These are your high-net-worth individual donors. They represent less than 1% of your data, but they wildly distort the mathematical averages (Mean is $17.47, which is higher than what 50% of people give, dragged up by the $10k outlier).

Campaign Fatigue (GiftTimeLast vs. GiftTimeFirst):

GiftTimeLast (median 18 months ago) is very tightly grouped (low kurtosis of 2.3).

GiftTimeFirst (median 64 months ago) has a negative kurtosis (-0.85). Negative kurtosis means a flatter, broader curve. This means the timeline of when people made their first donation is widely spread out over several years, rather than spiked at one specific event.

## Column Descriptions

### Identifier
| Column | Description |
|--------|-------------|
| **ID** | Unique donor identifier |

### Gift History (Last 36 Months)
| Column | Description |
|--------|-------------|
| **GiftCnt36** | Number of gifts given in the last 36 months |
| **GiftAvgLast** | Average value of the last gift (in dollars) |
| **GiftAvg36** | Average gift value over the last 36 months |
| **GiftCntCard36** | Number of gifts via credit card in the last 36 months |
| **GiftAvgCard36** | Average credit card gift value in the last 36 months |
| **GiftTimeLast** | Months since the last gift was received |

### Gift History (All Time)
| Column | Description |
|--------|-------------|
| **GiftCntAll** | Total number of gifts given (lifetime) |
| **GiftAvgAll** | Average gift value across all gifts (lifetime) |
| **GiftCntCardAll** | Total number of gifts via credit card (lifetime) |
| **GiftTimeFirst** | Months since the first gift was received |

### Promotion History (Campaign Contacts)
| Column | Description |
|--------|-------------|
| **PromCnt12** | Number of promotions/campaigns in the last 12 months |
| **PromCnt36** | Number of promotions/campaigns in the last 36 months |
| **PromCntAll** | Total number of promotions/campaigns (lifetime) |
| **PromCntCard12** | Number of credit card promotions in the last 12 months |
| **PromCntCard36** | Number of credit card promotions in the last 36 months |
| **PromCntCardAll** | Total number of credit card promotions (lifetime) |

### Demographic Information
| Column | Description |
|--------|-------------|
| **DemCluster** | Demographic cluster assignment (0-53) - donor segment classification |
| **DemAge** | Donor's age (in years) |
| **DemGender** | Donor's gender (M=Male, F=Female, U=Unknown) |
| **DemHomeOwner** | Home ownership status (H=Homeowner, U=Unknown) |
| **DemMedHomeValue** | Median home value in donor's neighborhood (in dollars) |
| **DemPctVeterans** | Percentage of veterans in donor's neighborhood |
| **DemMedIncome** | Median income in donor's neighborhood (in dollars) |

### Target Variables
| Column | Description |
|--------|-------------|
| **StatusCat96NK** | **PRIMARY TARGET** - Donor response status category (A, E, S, etc.) |
| **StatusCatStarAll** | **SECONDARY TARGET** - Binary indicator (0 or 1) for donor engagement |

---

### Key Insights by Category

**💰 Gift Metrics:** Track donor giving patterns - frequency, amount, and recency
- Higher gift counts and amounts indicate more engaged donors
- GiftTimeLast shows how recently a donor gave (lower = more recent)

**📧 Promotion Metrics:** Track campaign exposure and response
- PromCnt variables show how many times donors were contacted
- Useful for understanding campaign fatigue and engagement

**👤 Demographics:** Donor characteristics and neighborhood indicators
- DemCluster groups similar donors together for targeting
- Neighborhood metrics (income, home value, veterans %) provide socioeconomic context

## Target Variables Analysis

### Primary Target Variables

| Variable | Type | Description | Values | Purpose |
|----------|------|-------------|--------|---------|
| **Response** | Categorical | Did the donor respond to the campaign? | Yes / No | Classification target |
| **Donation_Amt** | Continuous | Amount donated (in dollars) | $0 - $500+ | Regression target |

### Key Observations

- **Response**: Binary classification target - predicting if a donor will respond
- **Donation_Amt**: Regression target - predicting the donation amount
  - Only available for donors who responded (Yes)
  - Missing values (NaN) for non-responders (No)
  - This is expected behavior in real-world fundraising data

### Data Quality Notes

- **Total Records**: 9,686 donors
- **Responders (Response='Yes')**: 4,843 (50%)
- **Non-Responders (Response='No')**: 4,843 (50%)
- **Donation_Amt Missing**: 4,843 (50%) - corresponds to non-responders

### Feature Engineering Columns

| Column | Description |
|--------|-------------|
| **Rep_DemMedIncome** | Replaced/imputed demographic median income |
| **IM_Rep_DemMedIncome** | Indicator flag for imputed income values |
| **log_GiftCnt36** | Log-transformed gift count (36 months) |

These engineered features are excluded from basic descriptive statistics as they are derived from other features.